### Query Enhancement – Query Expansion Techniques

In a RAG pipeline, the quality of the query sent to the retriever determines how good the retrieved context is — and therefore, how accurate the LLM’s final answer will be.

That’s where Query Expansion / Enhancement comes in.

#### 🎯 What is Query Enhancement?
Query enhancement refers to techniques used to improve or reformulate the user query to retrieve better, more relevant documents from the knowledge base.
It is especially useful when:

- The original query is short, ambiguous, or under-specified
- You want to broaden the scope to catch synonyms, related phrases, or spelling variants

In [7]:
from langchain_community.document_loaders import TextLoader
from langchain_classic.text_splitter import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain.chat_models import init_chat_model
from langchain_classic.prompts import PromptTemplate
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_classic.chains.retrieval import create_retrieval_chain
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableMap

In [12]:
## step1 : Load and split the dataset
loader = TextLoader("langchain_crewai_dataset.txt")
raw_documents = loader.load()
text_splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=50)
chunks = text_splitter.split_documents(raw_documents)

In [13]:
chunks

[Document(metadata={'source': 'langchain_crewai_dataset.txt'}, page_content='LangChain is an open-source framework designed for developing applications powered by large language models (LLMs). It simplifies the process of building, managing, and scaling complex chains of thought by abstracting prompt management, retrieval, memory, and agent orchestration. Developers can use'),
 Document(metadata={'source': 'langchain_crewai_dataset.txt'}, page_content='and agent orchestration. Developers can use LangChain to create end-to-end pipelines that connect LLMs with tools, APIs, vector databases, and other knowledge sources. (v1)'),
 Document(metadata={'source': 'langchain_crewai_dataset.txt'}, page_content='At the heart of LangChain lies the concept of chains, which are sequences of calls to LLMs and other tools. Chains can be simple, such as a single prompt fed to an LLM, or complex, involving multiple conditionally executed steps. LangChain makes it easy to compose and reuse chains using st

In [14]:
### step 2: Vector Store
embedding_model=HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
vectorstore=FAISS.from_documents(chunks,embedding_model)

## step 3:MMR Retriever
retriever=vectorstore.as_retriever(search_type="mmr",search_kwargs={"k":5})
retriever


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

VectorStoreRetriever(tags=['FAISS', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x000002A9AE684AD0>, search_type='mmr', search_kwargs={'k': 5})

In [16]:
import os
from groq import Groq

# Initialize client (Make sure GROQ_API_KEY is set in your environment variables)
client = Groq(api_key=os.environ.get("GROQ_API_KEY"))

# Fetch and print all active model IDs
models = client.models.list()
for model in models.data:
    print(model.id)

meta-llama/llama-prompt-guard-2-86m
openai/gpt-oss-120b
allam-2-7b
openai/gpt-oss-safeguard-20b
canopylabs/orpheus-v1-english
llama-3.1-8b-instant
llama-3.3-70b-versatile
whisper-large-v3-turbo
openai/gpt-oss-20b
whisper-large-v3
canopylabs/orpheus-arabic-saudi
groq/compound-mini
qwen/qwen3.6-27b
groq/compound
meta-llama/llama-prompt-guard-2-22m


In [20]:
## step 4 : LLM and Prompt

import os
from dotenv import load_dotenv
load_dotenv()

os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")

llm = init_chat_model("llama-3.3-70b-versatile", model_provider="groq")
llm


ChatGroq(metadata={'lc_versions': {'langchain-core': '1.5.4', 'langchain': '1.3.15'}}, profile={'name': 'Llama 3.3 70B Versatile', 'release_date': '2024-12-06', 'last_updated': '2024-12-06', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 32768, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x000002A9AE62D450>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000002A9AE62DE50>, model_name='llama-3.3-70b-versatile', model_kwargs={}, groq_api_key=SecretStr('**********'))

In [21]:
# Query expansion
query_expansion_prompt = PromptTemplate.from_template("""
You are a helpful assistant. Expand the following query to improve document retrieval by adding relevant synonyms, technical terms, and useful context.

Original query: "{query}"

Expanded query:
""")

query_expansion_chain=query_expansion_prompt| llm | StrOutputParser()
query_expansion_chain

PromptTemplate(input_variables=['query'], input_types={}, partial_variables={}, template='\nYou are a helpful assistant. Expand the following query to improve document retrieval by adding relevant synonyms, technical terms, and useful context.\n\nOriginal query: "{query}"\n\nExpanded query:\n')
| ChatGroq(metadata={'lc_versions': {'langchain-core': '1.5.4', 'langchain': '1.3.15'}}, profile={'name': 'Llama 3.3 70B Versatile', 'release_date': '2024-12-06', 'last_updated': '2024-12-06', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 32768, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x000002A9AE62D450>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0

In [23]:
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_classic.prompts import ChatPromptTemplate

answer_prompt = ChatPromptTemplate.from_template("""
Answer the question based only on the following context:

{context}

Question: {input}
""")

document_chain = create_stuff_documents_chain(llm, answer_prompt)

In [24]:
# Step 5: Full RAG pipeline with query expansion
rag_pipeline = (
    RunnableMap({
        "input": lambda x: x["input"],
        "context": lambda x: retriever.invoke(query_expansion_chain.invoke({"query": x["input"]}))
    })
    | document_chain
)

In [25]:
# Step 6: Run query
query = {"input": "What types of memory does LangChain support?"}
print(query_expansion_chain.invoke({"query":query}))
response = rag_pipeline.invoke(query)
print("✅ Answer:\n", response)

{'input': 'What types of memory does LangChain support?', 
 'expanded': 'What kinds of memory architectures, such as short-term memory, long-term memory, episodic memory, or working memory, are compatible with or utilized by LangChain? Are there any specific memory models, including recurrent neural network (RNN) memory, transformer memory, or external memory mechanisms, that LangChain is designed to work with or implement? Does LangChain provide support for memory-augmented neural networks, graph-based memory models, or other advanced memory techniques?'}
✅ Answer:
 LangChain supports two types of memory modules: 

1. ConversationBufferMemory
2. ConversationSummaryMemory

These allow the large language model (LLM) to maintain awareness of previous conversation turns or summarize long interactions to fit within token limits.


In [26]:
# Step 6: Run query
query = {"input": "What types of memory does LangChain support?"}
print(query_expansion_chain.invoke({"query":query}))
response = rag_pipeline.invoke(query)
print("✅ Answer:\n", response)

To improve document retrieval, I'll expand the original query by adding relevant synonyms, technical terms, and useful context. Here's the expanded query:

"{'input': 'What types of memory does LangChain support, including short-term memory, long-term memory, working memory, episodic memory, semantic memory, and procedural memory? Are there any specific memory architectures, such as recurrent neural networks (RNNs), long short-term memory (LSTM) networks, or transformer models, that LangChain is compatible with? Does LangChain utilize any particular memory management techniques, like caching, buffering, or attention mechanisms, to optimize performance? Are there any limitations or constraints on the types of memory that can be integrated with LangChain, and what are the requirements for implementing custom memory models or interfaces?'}"

This expanded query includes:

1. Synonyms for "memory" to capture different aspects of memory-related concepts.
2. Technical terms like "recurrent n